In [101]:
# 1) auto‑reload your .py changes
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [90]:
from fine_tuning.data_loader.load_wrf       import load_and_combine_wrf_files
from fine_tuning.data_loader.coord_util   import process_lat_lon
from fine_tuning.data_loader.pressure_util import process_pressure_levels
from fine_tuning.data_loader.atmos_util    import standardize_atmospheric_vars
from fine_tuning.data_loader.batch_util     import make_aurora_batch

In [91]:
import os

DATA_DIR = "/home/user/Documents/aurora/data_wrf"
meta, surf, static, atom = load_and_combine_wrf_files(
    os.path.join(DATA_DIR, "2d", "wrf2d_d01_2015-12-01_00:00:00.nc"),
    os.path.join(DATA_DIR, "2d", "wrf2d_d01_2015-12-01_03:00:00.nc"),
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_00:00:00.nc"),
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_03:00:00.nc"),
)

lat/ lon shapes: (1419,), (1429,)
surf shapes: {'t2': (2, 1419, 1429), 'u10': (2, 1419, 1429), 'v10': (2, 1419, 1429), 'psfc': (2, 1419, 1429)}
static shapes: {'z': (1419, 1429)}
atom shapes: {'z': (2, 51, 1419, 1429), 't': (2, 50, 1419, 1429), 'u': (2, 50, 1419, 1430), 'v': (2, 50, 1420, 1429), 'q': (2, 50, 1419, 1429)}
pressure_levels shape: (50, 1419, 1429)
dict_keys(['lat', 'lon', 'time', 'pressure_levels']) dict_keys(['t2', 'u10', 'v10', 'psfc']) dict_keys(['z']) dict_keys(['z', 't', 'u', 'v', 'q'])


In [92]:
# Diagnostics: raw meta / surf / static / atom
print("=== AFTER LOAD ===")
print("meta:")
for k,v in meta.items():
    print(f"  {k:15s}: type={type(v).__name__}, shape={getattr(v,'shape',None)}, dtype={getattr(v,'dtype',None)}")
print("\nsurf_vars:")
for k,v in surf.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")
print("\nstatic_vars:")
for k,v in static.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")
print("\natom_vars:")
for k,v in atom.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")

=== AFTER LOAD ===
meta:
  lat            : type=ndarray, shape=(1419,), dtype=float32
  lon            : type=ndarray, shape=(1429,), dtype=float32
  time           : type=ndarray, shape=(2,), dtype=|S19
  pressure_levels: type=ndarray, shape=(50, 1419, 1429), dtype=float32

surf_vars:
  t2             : shape=(2, 1419, 1429), dtype=float32
  u10            : shape=(2, 1419, 1429), dtype=float32
  v10            : shape=(2, 1419, 1429), dtype=float32
  psfc           : shape=(2, 1419, 1429), dtype=float32

static_vars:
  z              : shape=(1419, 1429), dtype=float32

atom_vars:
  z              : shape=(2, 51, 1419, 1429), dtype=float32
  t              : shape=(2, 50, 1419, 1429), dtype=float32
  u              : shape=(2, 50, 1419, 1430), dtype=float32
  v              : shape=(2, 50, 1420, 1429), dtype=float32
  q              : shape=(2, 50, 1419, 1429), dtype=float32


In [93]:
# 2) Fix coords
lat1d, lon1d = process_lat_lon(meta["lat"], meta["lon"])
meta["lat"], meta["lon"] = lat1d, lon1d

In [94]:
# Diagnostics: coords
print("\n=== AFTER COORDS PROCESSING ===")
print(f"lat1d.shape: {lat1d.shape}, decreasing? {all(lat1d[i]>lat1d[i+1] for i in range(len(lat1d)-1))}")
print(f"lon1d.shape: {lon1d.shape}, increasing? {all(lon1d[i]<lon1d[i+1] for i in range(len(lon1d)-1))}")



=== AFTER COORDS PROCESSING ===
lat1d.shape: (1419,), decreasing? True
lon1d.shape: (1429,), increasing? True


In [95]:
# 3) Process pressure levels
meta["pressure_levels"] = process_pressure_levels(meta["pressure_levels"], target=50)


In [96]:
# Diagnostics: pressure levels
pl = meta["pressure_levels"]
print("\n=== AFTER PRESSURE LEVELS ===")
print(f"pressure_levels: type={type(pl).__name__}, length={len(pl)}, min={pl[0]}, max={pl[-1]}")



=== AFTER PRESSURE LEVELS ===
pressure_levels: type=tuple, length=50, min=5141, max=105286


In [97]:
# 4) Standardize atmos shapes
H, W = surf["t2"].shape[1:]
atom = standardize_atmospheric_vars(atom, target_levels=50, target_y=H, target_x=W)


In [98]:
# Diagnostics: standardized atmos
print("\n=== AFTER ATMOS STANDARDIZATION ===")
for k,v in atom.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")


=== AFTER ATMOS STANDARDIZATION ===
  z              : shape=(2, 50, 1419, 1429), dtype=float32
  t              : shape=(2, 50, 1419, 1429), dtype=float32
  u              : shape=(2, 50, 1419, 1429), dtype=float32
  v              : shape=(2, 50, 1419, 1429), dtype=float32
  q              : shape=(2, 50, 1419, 1429), dtype=float32


In [102]:
# 5) Build the Aurora Batch
batch = make_aurora_batch(meta, surf, static, atom)


In [103]:

# Diagnostics: final Batch
print("\n=== FINAL BATCH ===")
print("surf_vars:")
for k,v in batch.surf_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nstatic_vars:")
for k,v in batch.static_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\natmos_vars:")
for k,v in batch.atmos_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nmetadata:")
print(f"  lat:           {tuple(batch.metadata.lat.shape)}")
print(f"  lon:           {tuple(batch.metadata.lon.shape)}")
print(f"  time:          {batch.metadata.time}")
print(f"  atmos_levels:  {batch.metadata.atmos_levels}")


=== FINAL BATCH ===
surf_vars:
  2t             : (1, 2, 1419, 1429), dtype=torch.float32
  10u            : (1, 2, 1419, 1429), dtype=torch.float32
  10v            : (1, 2, 1419, 1429), dtype=torch.float32
  msl            : (1, 2, 1419, 1429), dtype=torch.float32

static_vars:
  z              : (1419, 1429), dtype=torch.float32

atmos_vars:
  t              : (1, 2, 50, 1419, 1429), dtype=torch.float32
  u              : (1, 2, 50, 1419, 1429), dtype=torch.float32
  v              : (1, 2, 50, 1419, 1429), dtype=torch.float32
  q              : (1, 2, 50, 1419, 1429), dtype=torch.float32
  z              : (1, 2, 50, 1419, 1429), dtype=torch.float32

metadata:
  lat:           (1419,)
  lon:           (1429,)
  time:          (datetime.datetime(2015, 12, 1, 3, 0),)
  atmos_levels:  (np.int64(5141), np.int64(7185), np.int64(9229), np.int64(11273), np.int64(13316), np.int64(15360), np.int64(17404), np.int64(19448), np.int64(21491), np.int64(23535), np.int64(25579), np.int64(27623), 